**ISD-1020 · Master ISD (apprenticeship) · Université Paris-Saclay**

**Before any edit:** `File → Save a copy in Drive`.
Work only on **your** copy.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sklearn, torch

print("sklearn", sklearn.__version__, "| torch", torch.__version__)

# Course helpers — skip this

This cell is **not** an exercise: it downloads the plotting helpers and dataset
loaders if they are not already next to the notebook. You do not need to read
it. Continue below.


In [ ]:
# Environment setup (when run outside the repository, e.g. Google Colab)
import sys
import urllib.request
from pathlib import Path

_RAW_BASE = "https://raw.githubusercontent.com/stephane-rivaud/M2-ISD-ML-DL/main"

for _start in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_start / "pyproject.toml").is_file() and (_start / "common").is_dir():
        if str(_start) not in sys.path:
            sys.path.insert(0, str(_start))
        break

try:
    for module_path in ("common/__init__.py", "common/plots.py", "data/fetch.py"):
        local_file = Path(module_path)
        if local_file.is_file() or any(
            (Path(p) / module_path).is_file() for p in sys.path if p
        ):
            continue
        local_file.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{_RAW_BASE}/{module_path}", local_file)
except Exception:
    print(
        "Could not download the course helpers. Clone or download this repository so common/ and data/ sit next to this notebook."
    )


# Load forecasting — tomorrow at 18:00, how many MW?

ISD-1020 · Monday 28 September 2026 · morning lab (~140 min after the
40 min lecture, including optional fast track).

Thread of the module: **1. Problem · 2. Data · 3. Baseline · 4. Model · 5. Ablation ·
6. Error analysis → Recommendation**.
On 21 September: machine failures, leakages, boosting. Today:
**time series**, and the lie of a random validation.

In [ ]:
import random

from numpy.lib.stride_tricks import sliding_window_view
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from common.plots import plot_learning_curves
from data.fetch import load_eco2mix

RANDOM_STATE = 0
STEPS_PER_HOUR = 2
STEPS_PER_DAY = 48
STEPS_PER_WEEK = 336
H_1H = 2
H_24H = 48
WINDOW = 48
HGB_MAX_ITER = 80
MLP_WIDTH = 32
MLP_EPOCHS = 12
MLP_BATCH = 512
MLP_LR = 1e-3
GRU_HIDDEN = 32
GRU_EPOCHS = 8
GRU_BATCH = 256
# Validation slice inside the training period for learning curves only.
# The train/test cut itself is your decision later — pass it into helpers.
VAL_START = pd.Timestamp("2023-11-20")

# Metropolitan public holidays, 2023–2024. Listed explicitly so the notebook
# runs on Colab without an extra package. Easter Monday / Ascension / Whit
# Monday are the variable dates; the rest are fixed by statute.
HOLIDAYS = pd.to_datetime(
    [
        "2023-01-01",
        "2023-04-10",
        "2023-05-01",
        "2023-05-08",
        "2023-05-18",
        "2023-05-29",
        "2023-07-14",
        "2023-08-15",
        "2023-11-01",
        "2023-11-11",
        "2023-12-25",
        "2024-01-01",
        "2024-04-01",
        "2024-05-01",
        "2024-05-08",
        "2024-05-09",
        "2024-05-20",
        "2024-07-14",
        "2024-08-15",
        "2024-11-01",
        "2024-11-11",
        "2024-12-25",
    ]
)
HOLIDAY_DATES = set(HOLIDAYS.normalize().date)


def holiday_flag(index: pd.DatetimeIndex) -> np.ndarray:
    """1 on a metropolitan public holiday, else 0."""
    return np.array([ts.date() in HOLIDAY_DATES for ts in index], dtype=np.int8)


def mae_mape(y_true, y_pred) -> tuple[float, float]:
    """Return (MAE in MW, MAPE as a fraction)."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mae = float(np.mean(np.abs(y_true - y_pred)))
    mape = float(np.mean(np.abs(y_true - y_pred) / np.abs(y_true)))
    return mae, mape


def print_mae_mape(title: str, y_true, y_pred) -> tuple[float, float]:
    """Pretty-print MAE (MW) and MAPE, and return both."""
    mae, mape = mae_mape(y_true, y_pred)
    print(f"{title:32s}  MAE = {mae:7.1f} MW   MAPE = {mape:6.2%}")
    return mae, mape


def print_neighbour_in_train_share(n_rows: int, test_fraction: float = 0.2) -> float:
    """Share of a shuffled hold-out whose previous row index is still in train."""
    rng = np.random.RandomState(RANDOM_STATE)
    test_idx = rng.choice(n_rows, size=int(test_fraction * n_rows), replace=False)
    test_mask = np.zeros(n_rows, dtype=bool)
    test_mask[test_idx] = True
    nonzero = test_idx[test_idx > 0]
    share = float((~test_mask[nonzero - 1]).sum()) / len(test_idx)
    print(
        f"random {int((1 - test_fraction) * 100)}/{int(test_fraction * 100)}: "
        f"{share:.1%} of test points have their 30-min neighbour in train"
    )
    return share


def naive_copy(origin: pd.DatetimeIndex, horizon_steps: int, lag_steps: int) -> pd.Series:
    """Copy the series ``lag_steps`` before the target time, aligned on ``origin``."""
    target = origin + pd.Timedelta(minutes=30 * horizon_steps)
    return y.reindex(target - pd.Timedelta(minutes=30 * lag_steps))


def make_windows(horizon_steps: int):
    """Rows of ``WINDOW`` load values ending at the origin; target at origin+h."""
    values = y.to_numpy(dtype=np.float32)
    n = len(values)
    Xw = sliding_window_view(values, WINDOW)[: n - horizon_steps - WINDOW + 1]
    origins = y.index[WINDOW - 1 : n - horizon_steps]
    targets = values[WINDOW - 1 + horizon_steps : WINDOW - 1 + horizon_steps + len(Xw)]
    return Xw, targets, origins


def prepare_window_tensors(
    Xw,
    targets,
    origins,
    horizon_steps: int,
    cut: pd.Timestamp,
    val_start: pd.Timestamp = VAL_START,
):
    """Standardise on the training slice; return torch tensors and denorm stats.

    ``cut`` is the caller's train/test boundary (not chosen here). Training
    windows are those whose target time is still before ``cut`` and whose
    origin is before ``val_start``; validation keeps targets before ``cut``
    with origin at or after ``val_start``; test uses origins at or after ``cut``.
    """
    target_time = origins + pd.Timedelta(minutes=30 * horizon_steps)
    train_w = (target_time < cut) & (origins < val_start)
    val_w = (target_time < cut) & (origins >= val_start)
    test_w = origins >= cut
    print(
        "windows train/val/test:",
        int(train_w.sum()),
        int(val_w.sum()),
        int(test_w.sum()),
    )
    x_mean = float(Xw[train_w].mean())
    x_std = float(Xw[train_w].std())
    y_mean = float(targets[train_w].mean())
    y_std = float(targets[train_w].std())
    Xn = (Xw - x_mean) / x_std
    yn = (targets - y_mean) / y_std
    return {
        "X_train": torch.from_numpy(Xn[train_w]),
        "y_train": torch.from_numpy(yn[train_w]),
        "X_val": torch.from_numpy(Xn[val_w]),
        "y_val": torch.from_numpy(yn[val_w]),
        "X_test": torch.from_numpy(Xn[test_w]),
        "y_test_raw": targets[test_w],
        "y_mean": y_mean,
        "y_std": y_std,
    }


def train_window_model(model, epochs, batch_size, lr, X_train, y_train, X_val, y_val):
    """Adam + MSE on standardised windows. Returns per-epoch train/val MSE."""
    loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=batch_size,
        shuffle=True,
        generator=torch.Generator().manual_seed(RANDOM_STATE),
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    train_losses = []
    val_losses = []
    for _epoch in range(epochs):
        model.train()
        total = 0.0
        n = 0
        for xb, yb in loader:
            optimizer.zero_grad()
            pred = model(xb).squeeze(-1)
            loss = loss_fn(pred, yb)
            loss.backward()
            optimizer.step()
            total += loss.item() * yb.size(0)
            n += yb.size(0)
        train_losses.append(total / n)
        model.eval()
        with torch.no_grad():
            val_losses.append(float(loss_fn(model(X_val).squeeze(-1), y_val).item()))
    return train_losses, val_losses


def plot_two_horizon_week(
    target_index_short,
    y_true_short,
    y_pred_short,
    target_index_long,
    y_true_long,
    y_pred_long,
    week_start: pd.Timestamp,
    week_end: pd.Timestamp,
    short_title: str = "shorter horizon",
    long_title: str = "longer horizon",
):
    """Draw observed vs predicted for two horizons on one shared MW scale."""
    y_true_s = pd.Series(np.asarray(y_true_short), index=target_index_short).loc[
        week_start:week_end
    ]
    y_pred_s = pd.Series(np.asarray(y_pred_short), index=target_index_short).loc[
        week_start:week_end
    ]
    y_true_l = pd.Series(np.asarray(y_true_long), index=target_index_long).loc[
        week_start:week_end
    ]
    y_pred_l = pd.Series(np.asarray(y_pred_long), index=target_index_long).loc[
        week_start:week_end
    ]
    fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True, sharey=True)
    for ax, y_t, y_p, title in (
        (axes[0], y_true_s, y_pred_s, short_title),
        (axes[1], y_true_l, y_pred_l, long_title),
    ):
        ax.plot(y_t.index, y_t.to_numpy(), label="Observed")
        ax.plot(y_p.index, y_p.to_numpy(), label=title)
        ax.set_ylabel("MW")
        ax.set_title(title)
        ax.legend(loc="upper right")
        ax.grid(True, alpha=0.3)
    axes[1].set_xlabel("Target time")
    plt.tight_layout()
    plt.show()


def print_score_rows(rows):
    """Pretty-print (name, mae, mape) rows — display only."""
    print(f"{'model':28s}  {'MAE MW':>10s}  {'MAPE':>8s}")
    for name, mae, mape in rows:
        print(f"{name:28s}  {mae:10.1f}  {mape:8.2%}")


random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
print("seed", RANDOM_STATE)

## The problem — ⏱ ~6 min

RTE publishes **national electricity consumption** every 30 min
(éCO2mix). A forecaster, at time $t$, must state the load
in **1 h** and in **24 h**. Too low: risk on the
supply–demand balance. Too high: we start expensive plant for nothing.

The target is a **number** (MW), not a class. The metric that
matches the gap in MW is **MAE**; **MAPE** says the same
gap as a percentage of the load. Accuracy has no meaning here.

In pairs (30 s): in your company, which series would you want
to forecast at 24 h, and who pays for the error?

## Data — ⏱ ~8 min

RTE / ODRE, **national France** consumption, not Île-de-France.
Calendar years **2023–2024**, one point every 30 min. The file
is not in git: `load_eco2mix()` takes the local cache, otherwise
the ODRE API (a few seconds to a minute).

Licence **Ouverte v2.0 (Etalab)**. No temperature column: we will not
invent one.

In [ ]:
df = load_eco2mix()
y = df["Consommation (MW)"].astype(float)
print("shape:", df.shape)
print("columns:", list(df.columns))
print("index:", y.index.min(), "→", y.index.max())
print("slots/day:", df.groupby(y.index.date).size().unique())
print(f"mean = {y.mean():.0f} MW   min = {y.min():.0f}   max = {y.max():.0f}")
df.head()

35,088 points, 731 days × 48 steps, zero holes. Daylight-saving time has already
been cleaned by the loader (the source's empty :15 / :45 slots are
dropped). Both calendar years are available; how you separate past from
future for training is a decision you will take below. No subsampling —
the CPU budget allows the full series.

## Seasonality — ⏱ ~10 min

Two rhythms at a glance: the **week** (weekends lower) and
the **year** (winter high: heating; summer low, with air-conditioning
humps).

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharey=True)
for ax, start, title in (
    (axes[0], "2024-01-15", "Winter week — 15–21 January 2024"),
    (axes[1], "2024-07-15", "Summer week — 15–21 July 2024"),
):
    piece = y.loc[start : pd.Timestamp(start) + pd.Timedelta(days=7) - pd.Timedelta(minutes=30)]
    ax.plot(piece.index, piece.to_numpy())
    ax.set_title(title)
    ax.set_ylabel("MW")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

daily = y.resample("D").mean()
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(daily.index, daily.to_numpy())
ax.set_title("Daily mean — 2023–2024")
ax.set_ylabel("MW")
ax.grid(True, alpha=0.3)
plt.show()

**Exercise.** How smooth is national load across delays that matter for
forecasting? Pick a few delays (short, about a day, about a week). For
each, measure how tightly the series matches a delayed copy of itself,
and how large the error is if you simply reuse that delayed copy as a
prediction on the whole series. What does a very high short-delay match
imply for a shuffled validation?

In [ ]:
# To complete
...

**Take-away.** Neighbouring half-hours are almost the same value. That
measurement is **not** a day-ahead forecast: it only shows how smooth the
series is. A random split will therefore look far too easy.

## Legal features at origin $t$ — ⏱ ~18 min

An i.i.d. table can be shuffled. A series cannot: the **future** is the
test, the **past** is the train. If you shuffle, most hold-out points
still have their previous half-hour sitting in the train — and that
neighbour is almost a duplicate. The model interpolates; it does not
forecast.

Before any model: decide what a row may contain when the origin clock
reads $t$ and the label is the load at $t+h$. Calendar quantities that
are public for the **target** instant are not leakage. Load values that
would require peeking past $t$ are. The metropolitan holiday list and
the holiday helper above are available if you want a public-holiday bit.

Demo horizon for the next cells: **24 h**. scikit-learn's histogram
gradient boosting regressor is a reasonable tabular starter
(`max_iter` around 80, fixed seed).

**Exercise.** Write a builder that, given a forecast horizon in steps,
returns a feature table and a target series indexed by origin time.
Every feature must be knowable when the origin is $t$. Drop any row that
would need a load value you do not yet have. Print the column names and
the number of usable rows for the 24 h horizon.

In [ ]:
# To complete
...

## Cutting time — ⏱ ~12 min

Features alone do not make a validation honest. You still need a rule
that puts the **future** on one side and the **past** on the other, with
the model frozen before it sees the test origins. Training labels must
not sit in the period you pretend to hold out.

**Exercise.** Choose a calendar cut between the two years of data. Write
a function that, given origin timestamps and a horizon in steps, returns
boolean masks for training and for testing under that cut. Check that no
training target lands on or after the cut, and that test origins start
at the cut. Apply the masks to your 24 h table and print the two sizes.

In [ ]:
# To complete
...

## The shuffled lie — ⏱ ~12 min

Same features, same boosting family, two ways to allocate rows: a
shuffled hold-out versus the temporal masks you just built. The
neighbour-share helper above only *displays* how often a shuffled test
point still has its previous half-hour in train — it does not choose
your split.

**Exercise.** Train the same boosting setup twice on your 24 h table: once
with a shuffled hold-out, once with your temporal masks. Score MAE and
MAPE on each hold-out. Also run the neighbour-share helper on the number
of rows in your feature table. Which split looks kinder, and why?

In [ ]:
# To complete
...

**Take-away.** On honest features, the shuffled hold-out is clearly
kinder than the temporal one — here by roughly a fifth relative. That is
nothing like the **21 September** sub-flag leak (post-failure
`TWF`/`HDF`/`PWF`/`OSF`/`RNF` in `X`: PR-AUC **0.945** vs **~0.50**
without them): 2024 still looks like 2023, so a true future does not
explode. The gap is real because most shuffled test points keep their
previous half-hour in train. **We do not publish a shuffled-CV score on a
series.** Cite the temporal split.

## Naive baselines — ⏱ ~10 min

Before trusting boosting: copies of the past with **no learning**.

- **Persistence**: predict the load at $t+h$ by the load last seen at $t$.
- **Day-naive** (copying yesterday): same clock time one day earlier than
  the target.
- **Week-naive**: same clock time one week earlier than the target.

At 24 h, persistence and the day-naive coincide. A helper above builds
those copies once you choose the origin index, the horizon, and how far
before the target to look. Score on the **same test origins** you use for
the model — including a 1 h horizon built with the same feature and cut
rules.

**Exercise.** Reuse your feature builder and temporal masks for the 1 h
horizon as well as 24 h. On each test origin set, score persistence,
copying yesterday, and the week-naive with the helper defined above and
the MAE/MAPE printer. Keep the scores: they are the judges. A model that
loses against the best of the three has lost.

In [ ]:
# To complete
...

**Take-away.** On the temporal hold-out, copying yesterday beats the
week-naive at 24 h: weather sticks from one day to the next more than it
repeats week to week. The classic weekly seasonal is **not** the hardest
judge on *this* series. At 1 h the best no-learning judge is persistence;
at 24 h it is copying yesterday. Deploy only what beats that floor.

Mid-lab check (instructor circulating): each group can show a shuffled
score kinder than the temporal one, and three naive floors on the same
test origins. If boosting does not beat copying yesterday at 24 h, we do
not deploy it.

## HGB by horizon — ⏱ ~10 min

One boosting model per horizon — not a single net covering both. Same
feature family, same iteration budget, your temporal cut. You already
have the 24 h fit from the split comparison.

**Exercise.** Fit boosting on your 1 h temporal train set, predict on the
matching test origins, and compare MAE/MAPE to the best 1 h naive you
already scored. Does tabular boosting clear the floor at both horizons?

In [ ]:
# To complete
...

**Take-away.** On this protocol, boosting clears persistence at 1 h and
clears copying yesterday at 24 h. That is measured, not automatic.

## MLP on 48-step windows — ⏱ ~12 min

48 steps = **24 h** of load, flattened, no calendar. The network must
read the daily profile. Target horizon: **1 h**, so you can compare with
your tabular 1 h model.

A small feed-forward net with two hidden ReLU layers and a linear head
is enough for the slot. MSE, Adam, a dozen epochs, batch a few hundred.
Standardise window and target using **training** moments only.

Helpers above: build raw windows; prepare tensors given **your** cut (and
an optional validation start inside the train period); run the training
loop. They do not choose the cut for you.

**Exercise.** Using those helpers and **your** cut, prepare standardised
1 h window tensors. Define a compact MLP that maps one window to one
scalar. Train it with the training helper, plot the curves, then report
test MAE/MAPE next to your 1 h boosting score. Does the window net beat
the naive? Does it beat boosting?

In [ ]:
# To complete
...

**Take-away.** The window MLP usually beats persistence and still loses
to boosting on *this* series: boosting keeps lags **and** the public
holiday bit; the net sees raw load only. Clearing the naive justifies the
exercise; clearing the tabular baseline does not happen here.

## Errors by horizon — ⏱ ~6 min

Same winter test week, two horizons, one shared MW scale. Quoting a
short-horizon score in a day-ahead brief answers a different question.
Use the plotting helper and the score-table printer with objects you
already built.

**Exercise.** Overlay observed load and your boosting predictions for
both horizons on one mid-January week (shared vertical scale). Print a
small table of naive and model scores you already computed on the full
temporal test. Does the short horizon hug the profile while the long one
leaves a visible gap?

In [ ]:
# To complete
...

Read the graph: the short horizon **tracks** the intra-day profile; the
day-ahead panel leaves a wider gap on the **same** scale. Full-test
numbers in your table are the ones to cite — not a single flattering
short-horizon score pasted into a 24 h brief.

## Fast track — holidays, heat, optional GRU — ⏱ ~15 min

Two segments where copying yesterday either fails or is enough — then an
optional sequence model on the same windows as the MLP.

**Public holidays.** The metropolitan list is in the header; knowing the
**target** day is a holiday is public calendar knowledge.

**Heat waves.** The file has **no** temperature. A proxy only: in
June–August, days whose daily mean sits in the upper tail of that
summer distribution (lecture: high percentile of JJA daily means). Not a
Météo-France label.

In [ ]:
# ⚡ Fast track (optional) — for those who have already done ML
daily_mean = y.resample("D").mean()
jja = daily_mean[daily_mean.index.month.isin([6, 7, 8])]
heat_threshold = float(jja.quantile(0.90))
heat_days = set(jja[jja >= heat_threshold].index.date)
print(f"heat-wave proxy threshold (JJA daily mean) = {heat_threshold:.0f} MW")
print("proxy days:", len(heat_days))
print(*sorted(str(d) for d in heat_days), sep="\n")

**Exercise (optional).** On your 24 h temporal test:

1. Compare boosting vs copying yesterday on holiday targets versus the
   other days. Then refit the same boosting setup after dropping the
   holiday bit and repeat the holiday slice.
2. Compare boosting vs copying yesterday on your heat-proxy days versus
   the other June–August test days.

Where does boosting lose to the copy?

In [ ]:
# ⚡ Fast track (optional) — for those who have already done ML
# To complete
...

**Take-away.** With a holiday bit, boosting usually beats copying
yesterday on holiday targets, yet stays worse than on ordinary days —
few examples, odd profiles. Without the bit, holidays go bad and can lose
to the copy. On the heat-wave proxy, copying yesterday often wins: a hump
lasts several days. The proxy is **not** a temperature.

**Optional.** A GRU reads the window **in order**. Same standardised
tensors as the MLP, 1 h horizon, fewer epochs, CPU.

In [ ]:
# ⚡ Fast track (optional) — for those who have already done ML
class GRUForecaster(nn.Module):
    """One GRU layer, then a linear head on the last hidden state."""

    def __init__(self, hidden=GRU_HIDDEN):
        super().__init__()
        self.gru = nn.GRU(input_size=1, hidden_size=hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):
        # x: (batch, WINDOW) -> (batch, WINDOW, 1)
        _seq, hidden = self.gru(x.unsqueeze(-1))
        return self.head(hidden[-1]).squeeze(-1)


print(GRUForecaster())

**Exercise (optional).** Train the GRU on the same standardised window
tensors you prepared for the MLP (same cut, same 1 h horizon). Plot the
curves and compare test MAE/MAPE to your MLP, your 1 h boosting, and
persistence.

In [ ]:
# ⚡ Fast track (optional) — for those who have already done ML
# To complete
...

On a short CPU budget the GRU typically beats persistence and still loses
to both the MLP and boosting. Enough to open the door, not to replace the
tabular baseline.

## Attributions

- **éCO2mix** consolidated national consumption: RTE / [ODRE](https://odre.opendatasoft.com/explore/dataset/eco2mix-national-cons-def/),
  licence [Ouverte v2.0 (Etalab)](https://www.etalab.gouv.fr/licence-ouverte-open-licence/).
  2023–2024 dataset, 30 min steps, loaded by `data.fetch.load_eco2mix`.
- **Validation, boosting, lags**: [scikit-learn MOOC (Inria)](https://inria.github.io/scikit-learn-mooc/),
  modules 2 (*Selecting the best model*) and 7 (*Evaluating model
  performance*), licence [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
- **PyTorch training loop** (window → MLP / GRU): adapted
  from the [PyTorch tutorials](https://pytorch.org/tutorials/)
  (BSD-3-Clause).
- **Windows and sequence models**: [d2l.ai](https://d2l.ai/)
  ch. 8–9, licence [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).

## To go further at home

1. Hyndman & Athanasopoulos, *Forecasting: Principles and Practice*,
   ch. 5 (naives) and 5.10 (temporal CV):
   [otexts.com/fpp3](https://otexts.com/fpp3/)
2. Module 7 of the scikit-learn MOOC — *Evaluating model performance*.
3. d2l.ai, *Recurrent Neural Networks* :
   [d2l.ai/chapter_recurrent-neural-networks](https://d2l.ai/chapter_recurrent-neural-networks/index.html)

## Empirical protocol — ⏱ ~15 min

The six cells below are the thread of the module (and of the oral exam).
Answer in prose, tied to *this* notebook.

## Problem

Answer in prose (not only code).

1. Which **business decision** should the model inform, and what is the **target**?
2. Which **metric** matches that decision, and why not accuracy alone?
3. How do you separate training and test **without leakage** (split unit, time, stratification)?

## Data

Answer in prose (not only code).

1. What are the variables, their types, and any notable **imbalances** or volumes?
2. Which columns must you **not** use (identifiers, leakages, target sub-flags)?
3. What do you know about **quality** (missing values, outliers, drift) and temporal or business coverage?

## Baseline

Answer in prose (not only code).

1. Which **naive baseline** (majority class, mean, seasonal) and what score does it get?
2. Why is this baseline the **honest floor**, and not a "deliberately weak" model?
3. Which score must you **beat** to justify a more complex model?

## Model

Answer in prose (not only code).

1. Which **model family** do you choose, and why (not "because it is deep")?
2. How do you train it (split, validation, hyperparameters, loss)?
3. Does the model **beat the baseline** on the chosen metric, on an uncontaminated split?

## Ablation

Answer in prose (not only code).

1. What happens if you remove a family of variables, a regulariser, or a block of the network?
2. Which choice (features, architecture, horizon, threshold) **actually changes** the score?
3. Does the gain justify the extra **complexity** relative to the baseline?

## Error analysis → Recommendation

Answer in prose (not only code).

1. Where does the model go wrong (**segments**, error types, horizon)?
2. Are these errors **costly** for the business, and what would you change in the data or the model?
3. What concrete **recommendation**: deploy, do not deploy, stay on the baseline, or collect this specific data?